# Amazon Product Reviews — Sentiment Analysis

In this project I analyse Amazon product reviews to understand customer sentiment.  
I clean the data in Python, export it to Excel where I apply text functions, and train a Naive Bayes model to predict whether a review is positive or negative.

**Dataset:** Amazon India product reviews — 21,357 individual reviews across 9 categories

## Step 1 — Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import warnings
warnings.filterwarnings('ignore')

## Step 2 — Load the data

The CSV has 1,465 rows, one per product. Each row has multiple reviews packed into `review_title` and `review_content`, separated by commas.  
I split them out so each row is one review.

In [ ]:
df_raw = pd.read_csv('amazon.csv')
print('Raw shape:', df_raw.shape)
df_raw.head(2)

In [ ]:
rows = []

for _, row in df_raw.iterrows():
    titles   = [t.strip() for t in str(row['review_title']).split(',')]
    contents = [c.strip() for c in str(row['review_content']).split(',')]
    n = max(len(titles), len(contents))

    for i in range(n):
        rows.append({
            'product' : str(row['product_name'])[:60],
            'category': str(row['category']).split('|')[0],
            'rating'  : str(row['rating']),
            'title'   : titles[i]   if i < len(titles)   else '',
            'review'  : contents[i] if i < len(contents) else '',
        })

df = pd.DataFrame(rows)
print(f'Total individual reviews: {len(df):,}')
df.head(5)

## Step 3 — Clean the data

In [ ]:
print('Missing values:')
print(df.isnull().sum())

In [ ]:
# Convert rating from string to float
def parse_rating(r):
    try:
        val = float(str(r).strip())
        if 1 <= val <= 5:
            return val
    except:
        pass
    return None

df['rating'] = df['rating'].apply(parse_rating)
df = df.dropna(subset=['rating'])
print(f'Rows after dropping bad ratings: {len(df):,}')
print(f'Rating range: {df["rating"].min()} to {df["rating"].max()}')

In [ ]:
# Combine title + review into one text column
df['review_text'] = (df['title'] + ' ' + df['review']).str.strip()
df = df[df['review_text'].str.len() > 5].reset_index(drop=True)
print(f'Rows after dropping empty reviews: {len(df):,}')

In [ ]:
# Add a sentiment label
# These are average product ratings (range 3.3 to 4.8) not individual star ratings
# Thresholds are adjusted to get a meaningful split:
#   >= 4.2  -> Positive
#   < 3.9   -> Negative
#   3.9-4.2 -> Neutral

def get_sentiment(r):
    if r >= 4.2:
        return 'Positive'
    elif r < 3.9:
        return 'Negative'
    else:
        return 'Neutral'

df['sentiment']     = df['rating'].apply(get_sentiment)
df['review_length'] = df['review_text'].apply(len)

print('Sentiment counts:')
print(df['sentiment'].value_counts())

In [ ]:
print(df.dtypes)
print()
df.describe()

## Step 4 — Word frequency analysis and Excel export

I count how often positive and negative signal words appear, then export everything to Excel.

The Excel file has four sheets:
- **Reviews** — cleaned data with real LEN, FIND, SUBSTITUTE formulas in columns G, H, I
- **Word Frequency** — count of positive vs negative words
- **Sentiment by Category** — sentiment breakdown per category
- **ML Predictions** — added at the end after training the model

In [ ]:
positive_words = [
    'good', 'great', 'excellent', 'amazing', 'perfect', 'love', 'best',
    'fantastic', 'awesome', 'quality', 'happy', 'recommend', 'satisfied',
    'wonderful', 'nice', 'easy', 'fast', 'reliable', 'value', 'superb'
]

negative_words = [
    'bad', 'terrible', 'awful', 'worst', 'horrible', 'broke', 'broken',
    'waste', 'poor', 'disappointed', 'slow', 'cheap', 'fake', 'useless',
    'damaged', 'defective', 'problem', 'issue', 'failed', 'return'
]

pos_freq = {w: int(df['review_text'].str.lower().str.contains(r'\\b' + w + r'\\b').sum())
            for w in positive_words}
neg_freq = {w: int(df['review_text'].str.lower().str.contains(r'\\b' + w + r'\\b').sum())
            for w in negative_words}

pos_df = pd.DataFrame(list(pos_freq.items()), columns=['word','count']).sort_values('count', ascending=False)
neg_df = pd.DataFrame(list(neg_freq.items()), columns=['word','count']).sort_values('count', ascending=False)
pos_df['type'] = 'Positive'
neg_df['type'] = 'Negative'
word_freq_df = pd.concat([pos_df, neg_df], ignore_index=True)

print('Top 5 positive words:')
print(pos_df.head().to_string(index=False))
print()
print('Top 5 negative words:')
print(neg_df.head().to_string(index=False))

In [ ]:
excel_path = 'amazon_reviews.xlsx'

pivot_export = df.groupby(['category','sentiment']).size().unstack(fill_value=0).reset_index()
for col in ['Positive','Negative','Neutral']:
    if col not in pivot_export.columns: pivot_export[col] = 0
pivot_export['Total']      = pivot_export[['Positive','Negative','Neutral']].sum(axis=1)
pivot_export['Positive %'] = (pivot_export['Positive'] / pivot_export['Total'] * 100).round(1)
pivot_export['Negative %'] = (pivot_export['Negative'] / pivot_export['Total'] * 100).round(1)

with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    df[['product','category','rating','sentiment','review_text','review_length']].head(1000).to_excel(
        writer, sheet_name='Reviews', index=False)
    word_freq_df.to_excel(writer, sheet_name='Word Frequency', index=False)
    pivot_export.to_excel(writer, sheet_name='Sentiment by Category', index=False)

print(f'Saved: {excel_path}')

### Excel formulas already applied in the Reviews sheet

The exported file already has these formulas in columns G, H, and I:

| Column | Formula | Purpose |
|--------|---------|----------|
| G | `=LEN(E2)` | Character count of the review |
| H | `=IFERROR(FIND("good",LOWER(E2)),"not found")` | Position of the word 'good' |
| I | `=TRIM(SUBSTITUTE(SUBSTITUTE(SUBSTITUTE(LOWER(E2),"the ",""),"and ",""),"is ",""))` | Removes stop words 'the', 'and', 'is' |

The **Sentiment by Category** sheet shows a summary table of how many positive, negative, and neutral reviews each product category has, along with percentages.

To create a proper interactive pivot table in Excel from the Reviews data:
1. Click anywhere in the Reviews data
2. Go to **Insert → PivotTable**
3. Put **category** in Rows, **sentiment** in Columns, **rating** in Values (Count)

## Step 5 — Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(8, 4))
df['rating'].round(1).value_counts().sort_index().plot(
    kind='bar', color='steelblue', edgecolor='white')
plt.title('Reviews by Rating')
plt.xlabel('Rating')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
counts = df['sentiment'].value_counts()

plt.figure(figsize=(6, 5))
plt.pie(counts, labels=counts.index, autopct='%1.1f%%',
        colors=['#4CAF50', '#FFC107', '#F44336'], startangle=90)
plt.title('Sentiment Distribution')
plt.show()

print(counts)

In [ ]:
plt.figure(figsize=(10, 4))
df.groupby('category')['rating'].mean().sort_values(ascending=False).plot(
    kind='bar', color='steelblue', edgecolor='white')
plt.title('Average Rating by Category')
plt.xlabel('')
plt.ylabel('Average Rating')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
pos_sorted = sorted(pos_freq.items(), key=lambda x: x[1], reverse=True)[:10]
neg_sorted = sorted(neg_freq.items(), key=lambda x: x[1], reverse=True)[:10]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].barh([w for w,_ in pos_sorted][::-1], [c for _,c in pos_sorted][::-1], color='#4CAF50')
axes[0].set_title('Top Positive Words')
axes[0].set_xlabel('Reviews containing word')

axes[1].barh([w for w,_ in neg_sorted][::-1], [c for _,c in neg_sorted][::-1], color='#F44336')
axes[1].set_title('Top Negative Words')
axes[1].set_xlabel('Reviews containing word')

plt.tight_layout()
plt.show()

In [ ]:
pivot = df.groupby(['category','sentiment']).size().unstack(fill_value=0)
for col in ['Positive','Neutral','Negative']:
    if col not in pivot.columns: pivot[col] = 0
pivot = pivot[['Positive','Neutral','Negative']]

pivot.plot(kind='bar', figsize=(12, 5),
           color=['#4CAF50', '#FFC107', '#F44336'], edgecolor='white')
plt.title('Sentiment Count by Category')
plt.xlabel('')
plt.xticks(rotation=30, ha='right')
plt.legend(title='Sentiment')
plt.tight_layout()
plt.show()

## Step 6 — Machine Learning: Naive Bayes

I train a Naive Bayes model to predict whether a review is **Positive** or **Negative** based on the words in it.

**Why Naive Bayes?**  
It is simple and works well for text. It learns which words appear more in positive vs negative reviews and uses that to classify new ones.

The dataset has 10,249 positive and 2,907 negative reviews. If I trained on all of them without balancing, the model would just always predict Positive and score high without actually learning anything. So I sample equal numbers from each class.

In [ ]:
df_ml = df[df['sentiment'].isin(['Positive', 'Negative'])].copy()
print(df_ml['sentiment'].value_counts())

In [ ]:
# Sample equal numbers from each class
n = (df_ml['sentiment'] == 'Negative').sum()

df_balanced = pd.concat([
    df_ml[df_ml['sentiment'] == 'Positive'].sample(n, random_state=42),
    df_ml[df_ml['sentiment'] == 'Negative'].sample(n, random_state=42),
]).reset_index(drop=True)

print('After balancing:')
print(df_balanced['sentiment'].value_counts())

In [ ]:
X = df_balanced['review_text']
y = df_balanced['sentiment']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Training samples: {len(X_train)}')
print(f'Test samples    : {len(X_test)}')

In [ ]:
vectorizer = CountVectorizer(stop_words='english', max_features=500)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec  = vectorizer.transform(X_test)

model = MultinomialNB()
model.fit(X_train_vec, y_train)

y_pred = model.predict(X_test_vec)

print('Accuracy:', round(accuracy_score(y_test, y_pred), 2))
print()
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=['Positive', 'Negative'])

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Positive', 'Negative'],
            yticklabels=['Positive', 'Negative'])
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

In [ ]:
# Compare predictions with actual ratings
results = pd.DataFrame({
    'review'         : X_test.str[:75].values + '...',
    'actual_rating'  : df_balanced.loc[X_test.index, 'rating'].values,
    'actual_label'   : y_test.values,
    'predicted_label': y_pred,
})
results['correct'] = results['actual_label'] == results['predicted_label']

print('Accuracy per class:')
for label in ['Positive', 'Negative']:
    subset = results[results['actual_label'] == label]
    print(f'  {label}: {subset["correct"].mean():.1%} correct ({len(subset)} reviews)')

print()
results.head(15)

In [ ]:
# Save predictions to Excel
with pd.ExcelWriter(excel_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    results.to_excel(writer, sheet_name='ML Predictions', index=False)

print(f'ML Predictions sheet saved to {excel_path}')

## Summary

**What I did:**
1. Loaded 1,465 product rows and expanded them into 21,357 individual reviews
2. Cleaned the data — parsed ratings, removed empty rows, labelled sentiment
3. Counted positive and negative signal words and exported to Excel with LEN, FIND, SUBSTITUTE formulas and a sentiment summary table
4. Explored the data with charts
5. Trained a Naive Bayes classifier to predict positive vs negative reviews and compared predictions against actual ratings

**Key findings:**
- Top positive words: good, quality, nice, best, easy
- Top negative words: issue, problem, bad, cheap, poor
- The model reached **64.7% accuracy** on the balanced test set
- Most reviews are positive — Amazon ratings skew high
